# Delta-XAI & AdaptiveDeltaXAI — Google Colab Validation

**Paper:** Delta-XAI: A Unified Framework for Explaining Prediction Changes in Online Time Series Monitoring  
**Authors:** Kim et al. (AITRICS), 2025/2026  
**arXiv:** 2511.23036 | **OpenReview:** ZHW5pp5nE5  
**Related official repo:** [AITRICS/DeltaSHAP](https://github.com/AITRICS/DeltaSHAP) (workshop predecessor)

---

## ⚠️ IMPORTANT: Repository Status

| Repo | Status | Notes |
|------|--------|-------|
| `github.com/AITRICS/Delta-XAI` | **Does not exist** | URL returns 404. The paper is under review. |
| `github.com/AITRICS/DeltaSHAP` | ✅ Public | Workshop predecessor (ICMLW 2025). Built on WinIT. |
| `anonymous.4open.science/r/Delta-XAI` | 🔒 Read-only | Anonymous review submission, not clonable. |

**What this notebook does:** Validates the Delta-XAI concepts using:
1. The **DeltaSHAP** repo (the real, clonable AITRICS code — predecessor to Delta-XAI)
2. Our **AdaptiveDeltaXAI** reconstruction (faithful implementation of the paper's SWING mechanism)
3. Public datasets: **P19** (PhysioNet Sepsis 2019) + our **synthetic DC telemetry**

---
**Sections:**
```
A. Environment Setup
B. Clone & Audit DeltaSHAP (real AITRICS code)
C. Code Review: what DeltaSHAP does vs what Delta-XAI adds
D. Dataset: P19 PhysioNet (public, no credentials needed)
E. Dataset: Synthetic DC Temperature + Power telemetry
F. Run VanillaDeltaXAI (SWING baseline)
G. Run AdaptiveDeltaXAI (our enhanced wrapper)
H. Fidelity evaluation: AOPC + group-rank Spearman ρ
I. Results visualisation
J. Validation checklist
```

## A. Environment Setup

In [ ]:
# ── A1. Install dependencies ───────────────────────────────────────────────
# DeltaSHAP requirements (from their requirements.txt)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q captum shap lime scikit-learn pandas numpy matplotlib seaborn scipy tqdm
!pip install -q wget requests

print('✅ Packages installed')

In [ ]:
# ── A2. Core imports ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import torch.nn as nn
import time
import warnings
import os
import pickle
from collections import deque
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Tuple
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {DEVICE}')
print(f'   PyTorch: {torch.__version__}')

## B. Clone & Audit DeltaSHAP (Real AITRICS Code)

In [ ]:
# ── B1. Clone the real repo ────────────────────────────────────────────────
!git clone https://github.com/AITRICS/DeltaSHAP.git

# Verify the clone
!echo '\n── Repository structure:'
!find DeltaSHAP -type f | grep -v '__pycache__' | grep -v '.git' | sort

In [ ]:
# ── B2. Read and display the actual DeltaSHAP source files ─────────────────
import os

def show_file(path, max_lines=80):
    """Print file with line numbers."""
    if not os.path.exists(path):
        print(f'  [File not found: {path}]')
        return
    with open(path) as f:
        lines = f.readlines()
    total = len(lines)
    print(f'  [{path}  —  {total} lines total]')
    for i, line in enumerate(lines[:max_lines]):
        print(f'  {i+1:4d}│ {line}', end='')
    if total > max_lines:
        print(f'  ... ({total - max_lines} more lines)')

# Print key files
print('\n' + '='*70)
print('README.md')
print('='*70)
show_file('DeltaSHAP/README.md')

# List Python files in deltashap/
print('\n' + '='*70)
print('Python source files in deltashap/')
print('='*70)
for root, dirs, files in os.walk('DeltaSHAP/deltashap'):
    for f in sorted(files):
        if f.endswith('.py'):
            fpath = os.path.join(root, f)
            size = os.path.getsize(fpath)
            print(f'  {fpath}  ({size:,} bytes)')

In [ ]:
# ── B3. Show the core DeltaSHAP explanation logic ──────────────────────────
# Find files that contain the Shapley / explanation logic
import glob

py_files = glob.glob('DeltaSHAP/**/*.py', recursive=True)
print(f'Total Python files: {len(py_files)}')

# Show the main explanation file
key_keywords = ['shapley', 'shap', 'explain', 'delta', 'attribution', 'winit']

for fpath in sorted(py_files):
    with open(fpath) as f:
        content = f.read().lower()
    hits = [k for k in key_keywords if k in content]
    if hits:
        print(f'\n{fpath}  →  keywords: {", ".join(hits)}')
        show_file(fpath, max_lines=60)

## C. Code Review: DeltaSHAP vs Delta-XAI

### What DeltaSHAP does (the real AITRICS code)

DeltaSHAP computes **temporal Shapley values** that explain **changes** in risk predictions between consecutive timesteps. It is built on top of WinIT (Window-based Importance for Time series). Key mechanism:

```
ΔŷΔt = ŷ(t) - ŷ(t-1)          ← prediction change (the 'delta')

Attribution of ΔŷΔt to feature j:
φⱼ = Σ_{S⊆F\{j}} [|S|!(|F|-|S|-1)!/|F|!] · [v(S∪{j}) - v(S)]

where v(S) = E[Δŷ | features in S observed, rest marginalised]
```

### What Delta-XAI (arXiv:2511.23036) adds

Delta-XAI is the **generalisation and evaluation framework**:
- Wraps **14 XAI methods** (not just Shapley) under a unified delta-explanation protocol
- Introduces **SWING** (Shifted Window Integrated Gradients) — uses W_{t-1} as the IG baseline instead of zero
- Proposes a **new evaluation suite**: faithfulness (AOPC), sufficiency, and coherence metrics
- Key finding: classical IG outperforms dedicated methods when properly adapted

### What AdaptiveDeltaXAI adds on top
- Staleness-aware **caching** (skip SWING when explanation hasn't changed)
- Per-sensor **adaptive threshold** θᵢ = k·σᵢ(t)
- **Attribution drift** detector (second-order staleness)
- Online **detrending** (removes periodic DC workload components)
- **Group attribution** (rack zones / PDU grouping)

In [ ]:
# ── C1. Run DeltaSHAP's own scripts if data is available ───────────────────
# (DeltaSHAP uses MIMIC-III which requires PhysioNet credentials)
# We will instead use the P19 public dataset and synthetic DC data

# Show the run script so we understand expected I/O format
show_file('DeltaSHAP/scripts/run.sh', max_lines=50)
print()
show_file('DeltaSHAP/deltashap/__init__.py', max_lines=30)

## D. Dataset: P19 PhysioNet (Public, No Credentials Needed)

In [ ]:
# ── D1. Download pre-processed P19 (Sepsis Early Prediction 2019) ──────────
# Source: Raindrop paper's figshare (public, processed from PhysioNet)
# 34 sensors, 38,803 patients, binary sepsis label within 6h

os.makedirs('data/P19', exist_ok=True)

# Check if already downloaded
P19_URL = 'https://figshare.com/ndownloader/files/34673571'
P19_PATH = 'data/P19/P19data.zip'

if not os.path.exists('data/P19/PT_dict_list_6.npy'):
    print('Downloading P19 dataset (~50MB)...')
    !wget -q -O {P19_PATH} {P19_URL}
    !unzip -q {P19_PATH} -d data/P19/
    print('✅ P19 downloaded')
else:
    print('✅ P19 already present')

!ls -lh data/P19/

In [ ]:
# ── D2. Load and inspect P19 ───────────────────────────────────────────────
import numpy as np

# Try to load P19 — structure varies by source, handle both formats
try:
    P = np.load('data/P19/PT_dict_list_6.npy', allow_pickle=True)
    print(f'P19 loaded: {len(P)} patients')
    sample = P[0]
    print(f'Keys: {list(sample.keys())}')
    print(f'Time series shape: {sample["ts"].shape}')
    P19_LOADED = True
except Exception as e:
    print(f'P19 load issue: {e}')
    print('→ Will use synthetic proxy (same structure)')
    P19_LOADED = False

In [ ]:
# ── D3. Prepare P19 as streaming time series (or synthetic fallback) ────────

def prepare_p19_stream(P, n_patients=200, max_len=48, n_features=34):
    """
    Convert P19 patient dict list → numpy streaming format.
    Returns X (n_patients, max_len, n_features), y (n_patients,)
    """
    Xs, ys = [], []
    for p in P[:n_patients]:
        ts = p['ts']  # (T, features) irregular
        # Pad/truncate to max_len
        T = min(len(ts), max_len)
        x = np.zeros((max_len, n_features))
        x[-T:] = ts[:T, :n_features]
        Xs.append(x)
        ys.append(int(p.get('labels', [0])[-1]))
    return np.array(Xs), np.array(ys)


def make_synthetic_p19_proxy(n_patients=200, max_len=48, n_features=34, seed=42):
    """
    Synthetic proxy with same shape as P19 but generated.
    Use when P19 download is unavailable.
    """
    rng = np.random.RandomState(seed)
    X = rng.randn(n_patients, max_len, n_features).astype(np.float32)
    # Sepsis: 4% positive label
    y = (rng.rand(n_patients) < 0.04).astype(int)
    # Add anomaly signal in last 6 hours for positives
    for i in np.where(y == 1)[0]:
        onset = rng.randint(30, 42)
        X[i, onset:, :3] += rng.uniform(2, 5, (max_len - onset, 3))
    return X, y


if P19_LOADED:
    X_p19, y_p19 = prepare_p19_stream(P, n_patients=200)
    print(f'P19 stream: X={X_p19.shape}, y={y_p19.shape}, pos_rate={y_p19.mean():.2%}')
else:
    X_p19, y_p19 = make_synthetic_p19_proxy()
    print(f'Synthetic P19 proxy: X={X_p19.shape}, y={y_p19.shape}, pos_rate={y_p19.mean():.2%}')

## E. Dataset: Synthetic DC Telemetry (Temperature + Power)

In [ ]:
# ── E1. DC Temperature generator ──────────────────────────────────────────

def generate_dc_temperature(
    n_sensors=8, duration_min=120, freq_s=10,
    n_anomaly_bursts=4, burst_len_s=120, seed=42
):
    """
    Physical model:
      temp_i(t) = baseline_i + diurnal(t) + cooling_cycle(t)
                + thermal_coupling(t) + noise + anomaly_burst(t)

    baseline_i  = 35°C + U(-2,+2)         sensor-specific offset
    diurnal(t)  = 2·sin(2πt/86400)         24h cycle
    cooling(t)  = 1.5·sin(2πt/600 + φ_i)  10-min CRAC cycle
    coupling(t) = 0.15·temp_{i-1}(t)       left-neighbour heat transfer
    anomaly     = +U(8,20)°C for burst_len_s seconds
    """
    rng = np.random.RandomState(seed)
    n = int(duration_min * 60 / freq_s)
    t = np.arange(n) * freq_s
    data = {}
    labels = np.zeros(n, dtype=int)

    # Anomaly burst windows
    burst_starts = rng.choice(np.arange(n//4, 3*n//4),
                               n_anomaly_bursts, replace=False)
    burst_len = burst_len_s // freq_s
    for bs in burst_starts:
        labels[bs: min(bs + burst_len, n)] = 1

    for i in range(n_sensors):
        diurnal  = 2.0 * np.sin(2*np.pi*t/86400 + i*0.3)
        cooling  = 1.5 * np.sin(2*np.pi*t/600 + i*0.5)
        baseline = 35.0 + rng.uniform(-2, 2)
        noise    = rng.randn(n) * 0.3
        temp     = baseline + diurnal + cooling + noise
        if i > 0:  # thermal coupling
            temp += 0.15 * data[f'temp_{i-1}']
        spike_mag = rng.uniform(8, 20, n)
        temp += spike_mag * labels
        data[f'temp_{i}'] = temp

    df = pd.DataFrame(data)
    df['timestamp'] = pd.date_range('2025-01-15 08:00', periods=n, freq=f'{freq_s}s')
    df['anomaly']   = labels
    return df


def generate_dc_power(
    n_servers=8, duration_min=120, freq_s=10,
    n_anomaly_bursts=4, burst_len_s=90, seed=42
):
    """
    Physical model:
      power_i(t) = baseline_i + workload_cycle(t) + PDU_coupling(t)
                 + noise + anomaly_surge(t)

    baseline_i  = 200W + U(-20,+20)
    workload(t) = 50·sin(2πt/1800 + φ_i)  30-min batch cycle
    PDU(t)      = 20·sin(2πt/1800) + N(0,3²)  shared per PDU pair
    anomaly     = power × U(1.5, 2.5)  runaway process / PSU fault
    """
    rng = np.random.RandomState(seed + 10)
    n = int(duration_min * 60 / freq_s)
    t = np.arange(n) * freq_s
    data = {}
    labels = np.zeros(n, dtype=int)

    burst_starts = rng.choice(np.arange(n//4, 3*n//4),
                               n_anomaly_bursts, replace=False)
    burst_len = burst_len_s // freq_s
    for bs in burst_starts:
        labels[bs: min(bs + burst_len, n)] = 1

    pdu = {p: 20*np.sin(2*np.pi*t/1800 + p*1.1) + rng.randn(n)*3
           for p in range(n_servers//2)}

    for i in range(n_servers):
        workload = 50 * np.sin(2*np.pi*t/1800 + i*0.8)
        baseline = 200 + rng.uniform(-20, 20)
        noise    = rng.randn(n) * 4
        power    = baseline + workload + pdu[i//2] + noise
        surge    = rng.uniform(1.5, 2.5, n)
        power    = np.where(labels == 1, power * surge, power)
        data[f'power_{i}'] = np.clip(power, 50, 900)

    df = pd.DataFrame(data)
    df['timestamp'] = pd.date_range('2025-01-15 08:00', periods=n, freq=f'{freq_s}s')
    df['anomaly']   = labels
    return df


# Generate both datasets
print('Generating DC datasets...')
df_temp  = generate_dc_temperature(n_sensors=8, duration_min=120,
                                    n_anomaly_bursts=4)
df_power = generate_dc_power(n_servers=8, duration_min=120,
                              n_anomaly_bursts=4)

temp_cols  = [c for c in df_temp.columns  if c.startswith('temp_')]
power_cols = [c for c in df_power.columns if c.startswith('power_')]

stream_temp  = df_temp[temp_cols].values.astype(np.float32)
stream_power = df_power[power_cols].values.astype(np.float32)
stream_power_norm = stream_power / stream_power.max()

print(f'Temperature: {stream_temp.shape}  | anomaly_rate={df_temp.anomaly.mean():.1%}')
print(f'Power:       {stream_power.shape} | anomaly_rate={df_power.anomaly.mean():.1%}')

In [ ]:
# ── E2. Visualise both datasets ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('Synthetic DC Telemetry — Dataset Validation', fontsize=14, fontweight='bold')

# Temperature: all sensors + anomaly bands
ax = axes[0, 0]
for col in temp_cols:
    ax.plot(df_temp['timestamp'], df_temp[col], lw=0.6, alpha=0.7)
# Shade anomaly windows
in_anom = False
for i, (ts, a) in enumerate(zip(df_temp['timestamp'], df_temp['anomaly'])):
    if a == 1 and not in_anom:
        start_ts = ts; in_anom = True
    elif a == 0 and in_anom:
        ax.axvspan(start_ts, ts, color='red', alpha=0.15)
        in_anom = False
ax.set_title('Temperature (°C) — 8 sensors, 4 anomaly bursts')
ax.set_ylabel('°C'); ax.tick_params(axis='x', rotation=30)

# Temperature: anomaly zoom
ax = axes[0, 1]
zoom_mask = (df_temp['anomaly'] == 1)
zoom_idx  = np.where(zoom_mask)[0]
if len(zoom_idx) > 0:
    z_start = max(0, zoom_idx[0] - 20)
    z_end   = min(len(df_temp), zoom_idx[0] + 80)
    df_z    = df_temp.iloc[z_start:z_end]
    for col in temp_cols:
        ax.plot(df_z['timestamp'], df_z[col], lw=1.0, alpha=0.8)
    anom_z = df_z[df_z['anomaly'] == 1]
    if len(anom_z) > 0:
        ax.axvspan(anom_z['timestamp'].iloc[0], anom_z['timestamp'].iloc[-1],
                   color='red', alpha=0.2, label='Anomaly')
ax.set_title('Temperature — anomaly burst zoom'); ax.tick_params(axis='x', rotation=30)

# Power: all servers
ax = axes[1, 0]
for col in power_cols:
    ax.plot(df_power['timestamp'], df_power[col], lw=0.6, alpha=0.7)
in_anom = False
for i, (ts, a) in enumerate(zip(df_power['timestamp'], df_power['anomaly'])):
    if a == 1 and not in_anom:
        start_ts = ts; in_anom = True
    elif a == 0 and in_anom:
        ax.axvspan(start_ts, ts, color='red', alpha=0.15)
        in_anom = False
ax.set_title('Power (W) — 8 servers, 4 anomaly bursts')
ax.set_ylabel('Watts'); ax.tick_params(axis='x', rotation=30)

# Power: anomaly zoom
ax = axes[1, 1]
zoom_mask = (df_power['anomaly'] == 1)
zoom_idx  = np.where(zoom_mask)[0]
if len(zoom_idx) > 0:
    z_start = max(0, zoom_idx[0] - 20)
    z_end   = min(len(df_power), zoom_idx[0] + 80)
    df_z    = df_power.iloc[z_start:z_end]
    for col in power_cols:
        ax.plot(df_z['timestamp'], df_z[col], lw=1.0, alpha=0.8)
    anom_z = df_z[df_z['anomaly'] == 1]
    if len(anom_z) > 0:
        ax.axvspan(anom_z['timestamp'].iloc[0], anom_z['timestamp'].iloc[-1],
                   color='red', alpha=0.2)
ax.set_title('Power — anomaly burst zoom'); ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('dc_telemetry_datasets.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dataset plot saved')

## F. VanillaDeltaXAI — SWING Baseline (Paper-Faithful)

In [ ]:
# ── F1. GRU model for DC telemetry (matches DeltaSHAP architecture) ────────
class GRUAnomalyModel(nn.Module):
    """
    Single-layer GRU anomaly scorer.
    Matches the architecture used in DeltaSHAP (gru1layer).
    Input:  (batch, window, features)
    Output: scalar anomaly score in [0, 1]
    """
    def __init__(self, n_features, hidden=32):
        super().__init__()
        self.gru  = nn.GRU(n_features, hidden, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        _, h = self.gru(x)
        return self.head(h.squeeze(0)).squeeze(-1)


def make_model_fn(model: nn.Module) -> Callable:
    """Wrap PyTorch model as numpy callable for XAI."""
    model.eval()
    def fn(W: np.ndarray) -> float:
        # W: (1, window, features)
        with torch.no_grad():
            x = torch.from_numpy(W.astype(np.float32))
            return float(model(x).item())
    return fn


# Instantiate models
N_FEAT_TEMP  = stream_temp.shape[1]
N_FEAT_POWER = stream_power_norm.shape[1]

gru_temp  = GRUAnomalyModel(N_FEAT_TEMP).to(DEVICE)
gru_power = GRUAnomalyModel(N_FEAT_POWER).to(DEVICE)

model_temp  = make_model_fn(gru_temp)
model_power = make_model_fn(gru_power)

# Quick sanity check
test_batch_temp  = stream_temp[:1].reshape(1, 1, N_FEAT_TEMP)
test_batch_power = stream_power_norm[:1].reshape(1, 1, N_FEAT_POWER)
print(f'GRU Temp  output: {model_temp(test_batch_temp):.4f}')
print(f'GRU Power output: {model_power(test_batch_power):.4f}')
print(f'✅ Models initialised (untrained — for explanation mechanism validation)')

In [ ]:
# ── F2. VanillaDeltaXAI (SWING — paper-faithful) ───────────────────────────

class VanillaDeltaXAI:
    """
    Faithful reproduction of Delta-XAI SWING mechanism.

    SWING formula (Kim et al., 2026):
        E_t = (W_t - W_{t-1}) * (1/m) * Σ_{k=1}^{m} ∇f(W_{t-1} + k/m · ΔW)

    This is Integrated Gradients with the SHIFTED baseline:
    - Standard IG:   baseline = zero vector
    - SWING:         baseline = W_{t-1} (previous window)

    The key insight: using W_{t-1} as baseline means the attribution
    quantifies the CONTRIBUTION OF THE CHANGE ΔW, not of W_t itself.
    This is the fundamental delta-explanation property.
    """

    def __init__(self, model, window=30, n_steps=20, eps=1e-4):
        self.model  = model
        self.w      = window
        self.m      = n_steps
        self.eps    = eps
        self.buffer = deque(maxlen=window)
        self.W_prev = None
        self.t      = 0

    def _build_window(self, x_t):
        self.buffer.append(x_t.copy())
        W = np.array(self.buffer)
        if len(W) < self.w:
            W = np.vstack([np.zeros((self.w - len(W), x_t.shape[0])), W])
        return W

    def _gradient(self, W):
        """Numerical gradient ∇f(W) via finite differences."""
        base = self.model(W[np.newaxis])
        G    = np.zeros_like(W)
        for i in range(W.shape[0]):
            for j in range(W.shape[1]):
                W2       = W.copy()
                W2[i, j] += self.eps
                G[i, j]  = (self.model(W2[np.newaxis]) - base) / self.eps
        return G

    def _swing(self, W_prev, W_curr):
        """SWING: piecewise IG from W_prev → W_curr."""
        dW   = W_curr - W_prev
        grads = [self._gradient(W_prev + (k/self.m) * dW)
                 for k in range(1, self.m + 1)]
        return dW * np.mean(grads, axis=0)  # completeness axiom holds

    def update(self, x_t):
        t0     = time.perf_counter()
        W_curr = self._build_window(x_t)
        self.t += 1
        if self.W_prev is None:
            E = np.zeros_like(W_curr)
        else:
            E = self._swing(self.W_prev, W_curr)
        self.W_prev = W_curr.copy()
        return E, (time.perf_counter() - t0) * 1000


print('✅ VanillaDeltaXAI (SWING) defined')

In [ ]:
# ── F3. Run SWING baseline on both streams ─────────────────────────────────
WINDOW   = 20
N_STEPS  = 5    # IG steps (use 20+ for publication quality)
N_EVAL   = 120  # steps to evaluate

print('Running VanillaDeltaXAI baseline...')

def run_vanilla(stream, model_fn, label):
    vanilla = VanillaDeltaXAI(model=model_fn, window=WINDOW, n_steps=N_STEPS)
    Es, lats, preds = [], [], []
    for i, x in enumerate(stream[:N_EVAL]):
        E, lat = vanilla.update(x)
        W = np.array(vanilla.buffer)
        if len(W) < WINDOW:
            W = np.vstack([np.zeros((WINDOW-len(W), x.shape[0])), W])
        pred = model_fn(W[np.newaxis])
        Es.append(E); lats.append(lat); preds.append(pred)
    print(f'  [{label}] mean latency={np.mean(lats):.2f}ms  '
          f'p95={np.percentile(lats, 95):.2f}ms  '
          f'mean|E|={np.mean([np.abs(e).mean() for e in Es]):.5f}')
    return Es, lats, preds

E_van_temp,  lat_van_temp,  pred_van_temp  = run_vanilla(stream_temp,       model_temp,  'Temperature')
E_van_power, lat_van_power, pred_van_power = run_vanilla(stream_power_norm, model_power, 'Power')

## G. AdaptiveDeltaXAI — Enhanced Wrapper

In [ ]:
# ── G1. AdaptiveDeltaXAI class ─────────────────────────────────────────────

@dataclass
class ExplanationResult:
    t: int
    E: np.ndarray
    triggered: bool
    reason: str
    latency_ms: float
    cache_age: int
    delta_pred: float


class AdaptiveDeltaXAI:
    """
    Extends Delta-XAI (Kim et al., 2026) with:
      1. Staleness-aware caching (θᵢ = k·σᵢ per sensor)
      2. Attribution drift detector (||E - EMA(E)||_F > ψ)
      3. Online EMA detrending (isolates anomaly-relevant residuals)
      4. Temporal discount (γ^lag weighting)
      5. Optional group attribution (ShaTS-style)
    """

    def __init__(self, model, window=30, n_steps=5, eps=1e-4,
                 k_sigma=2.5, t_max=100, gamma=0.97,
                 alpha_ema=0.05, psi=0.30,
                 feature_groups=None, name='AdaptiveDeltaXAI'):
        self.model   = model
        self.w       = window
        self.m       = n_steps
        self.eps     = eps
        self.k       = k_sigma
        self.t_max   = t_max
        self.gamma   = gamma
        self.alpha   = alpha_ema
        self.psi     = psi
        self.groups  = feature_groups or {}
        self.name    = name

        self.buffer    = deque(maxlen=window)
        self.W_prev    = None
        self.E_cache   = None
        self.E_ema     = None
        self.sigma_ema = None
        self.trend_ema = None
        self.pred_prev = None
        self.t = 0;  self.t_last = -t_max
        self.log: List[ExplanationResult] = []

    def _build_window(self, x_t):
        self.buffer.append(x_t.copy())
        W = np.array(self.buffer)
        if len(W) < self.w:
            W = np.vstack([np.zeros((self.w - len(W), x_t.shape[0])), W])
        return W

    def _detrend(self, x_t):
        if self.trend_ema is None:
            self.trend_ema = x_t.copy()
        self.trend_ema = (1 - self.alpha)*self.trend_ema + self.alpha*x_t
        return x_t - self.trend_ema

    def _trigger(self, residual):
        age = self.t - self.t_last
        if age >= self.t_max:         return True, 'max_age'
        if self.E_cache is None:      return True, 'init'

        # Per-sensor adaptive θᵢ
        delta = np.abs(residual)
        if self.sigma_ema is None:
            self.sigma_ema = delta + 1e-6
        self.sigma_ema = (1-self.alpha)*self.sigma_ema + self.alpha*delta
        if np.any(delta > self.k * self.sigma_ema):
            return True, 'pred_delta'

        # Attribution drift
        if self.E_ema is not None:
            if np.linalg.norm(self.E_cache - self.E_ema, 'fro') > self.psi:
                return True, 'attr_drift'

        return False, 'cached'

    def _grad(self, W):
        base = self.model(W[np.newaxis])
        G    = np.zeros_like(W)
        for i in range(W.shape[0]):
            for j in range(W.shape[1]):
                W2       = W.copy()
                W2[i, j] += self.eps
                G[i, j]  = (self.model(W2[np.newaxis]) - base) / self.eps
        return G

    def _swing(self, W_prev, W_curr):
        if self.groups:
            return self._group_swing(W_prev, W_curr)
        dW    = W_curr - W_prev
        grads = [self._grad(W_prev + (k/self.m)*dW) for k in range(1, self.m+1)]
        E     = dW * np.mean(grads, axis=0)
        return self._discount(E)

    def _group_swing(self, W_prev, W_curr):
        E      = np.zeros_like(W_curr)
        p_full = self.model(W_curr[np.newaxis])
        for grp, idxs in self.groups.items():
            Wm           = W_curr.copy()
            Wm[:, idxs]  = W_prev[:, idxs]
            contrib      = (p_full - self.model(Wm[np.newaxis])) / max(len(idxs), 1)
            E[:, idxs]   = contrib
        return self._discount(E)

    def _discount(self, E):
        lags = np.arange(E.shape[0] - 1, -1, -1)
        return E * (self.gamma**lags)[:, np.newaxis]

    def update(self, x_t):
        t0      = time.perf_counter()
        self.t += 1
        W_curr  = self._build_window(x_t)
        residual = self._detrend(x_t)
        pred     = self.model(W_curr[np.newaxis])

        triggered, reason = self._trigger(residual)
        if triggered:
            if self.W_prev is None:
                self.E_cache = np.zeros_like(W_curr)
            else:
                self.E_cache = self._swing(self.W_prev, W_curr)
            if self.E_ema is None:
                self.E_ema = self.E_cache.copy()
            self.E_ema   = 0.9*self.E_ema + 0.1*self.E_cache
            self.t_last  = self.t

        delta_pred = abs(pred - self.pred_prev) if self.pred_prev is not None else 0.0
        self.W_prev    = W_curr.copy()
        self.pred_prev = pred

        r = ExplanationResult(
            t=self.t, E=self.E_cache.copy() if self.E_cache is not None
              else np.zeros_like(W_curr),
            triggered=triggered, reason=reason,
            latency_ms=(time.perf_counter()-t0)*1000,
            cache_age=self.t - self.t_last,
            delta_pred=delta_pred
        )
        self.log.append(r)
        return r

    def summary(self):
        if not self.log: return {}
        n     = len(self.log)
        n_rc  = sum(r.triggered for r in self.log)
        lats  = [r.latency_ms for r in self.log]
        ages  = [r.cache_age  for r in self.log]
        rsns  = {}
        for r in self.log:
            if r.triggered:
                rsns[r.reason] = rsns.get(r.reason, 0) + 1
        return dict(n=n, n_recompute=n_rc,
                    recompute_pct=100*n_rc/n,
                    lat_mean=np.mean(lats), lat_p95=np.percentile(lats,95),
                    lat_p99=np.percentile(lats,99),
                    cache_age_mean=np.mean(ages), cache_age_max=int(np.max(ages)),
                    trigger_reasons=rsns)

print('✅ AdaptiveDeltaXAI defined')

In [ ]:
# ── G2. Run AdaptiveDeltaXAI on both streams ───────────────────────────────

# Temperature: rack zone groups
temp_groups  = {'zone_A': [0,1,2,3], 'zone_B': [4,5,6,7]}
# Power: PDU groups
power_groups = {'PDU_1':  [0,1,2,3], 'PDU_2':  [4,5,6,7]}

adx_temp = AdaptiveDeltaXAI(
    model=model_temp, window=WINDOW, n_steps=N_STEPS,
    k_sigma=2.5, t_max=60, gamma=0.97, alpha_ema=0.05, psi=0.25,
    feature_groups=temp_groups, name='AdaptiveDeltaXAI-Temp'
)

adx_power = AdaptiveDeltaXAI(
    model=model_power, window=WINDOW, n_steps=N_STEPS,
    k_sigma=3.0, t_max=60, gamma=0.95, alpha_ema=0.08, psi=0.20,
    feature_groups=power_groups, name='AdaptiveDeltaXAI-Power'
)

print('Running AdaptiveDeltaXAI...')

results_temp  = [adx_temp.update(x)  for x in stream_temp[:N_EVAL]]
results_power = [adx_power.update(x) for x in stream_power_norm[:N_EVAL]]

sm_t = adx_temp.summary()
sm_p = adx_power.summary()

print(f'\nTemperature:')
print(f'  Recompute rate: {sm_t["recompute_pct"]:.1f}%  | mean latency: {sm_t["lat_mean"]:.3f}ms')
print(f'  Cache age mean: {sm_t["cache_age_mean"]:.1f}  max: {sm_t["cache_age_max"]}')
print(f'  Trigger reasons: {sm_t["trigger_reasons"]}')
print(f'\nPower:')
print(f'  Recompute rate: {sm_p["recompute_pct"]:.1f}%  | mean latency: {sm_p["lat_mean"]:.3f}ms')
print(f'  Cache age mean: {sm_p["cache_age_mean"]:.1f}  max: {sm_p["cache_age_max"]}')
print(f'  Trigger reasons: {sm_p["trigger_reasons"]}')

## H. Fidelity Evaluation: AOPC + Group-Rank Spearman ρ

In [ ]:
# ── H1. AOPC (Area Over the Perturbation Curve) ───────────────────────────
# This is the paper's primary faithfulness metric
# Procedure: rank features by |E|, mask top-k one by one,
# measure prediction degradation. Higher AOPC = more faithful.

def aopc(model_fn, W_curr, E, k_max=None, mask_val=0.0):
    """
    Area Over the Perturbation Curve.
    W_curr: (window, features)
    E:      (window, features) attribution matrix
    Returns: AOPC score (scalar)
    """
    if k_max is None:
        k_max = min(W_curr.shape[0] * W_curr.shape[1] // 4, 20)

    baseline_pred = model_fn(W_curr[np.newaxis])

    # Rank (lag, feat) pairs by absolute attribution, descending
    flat_E    = np.abs(E).flatten()
    ranked    = np.argsort(-flat_E)  # descending

    W_masked  = W_curr.copy()
    degradations = []

    for k in range(1, k_max + 1):
        idx     = ranked[k - 1]
        row, col = np.unravel_index(idx, E.shape)
        W_masked[row, col] = mask_val
        masked_pred = model_fn(W_masked[np.newaxis])
        degradations.append(baseline_pred - masked_pred)

    return float(np.mean(degradations))


# ── H2. Group-Rank Spearman ρ ─────────────────────────────────────────────
def group_rank_spearman(E_adaptive, E_vanilla, groups):
    """
    Compute Spearman ρ between group-level attributions.
    Correctly handles grouped vs full-feature attribution mismatch.
    E_adaptive: (window, features) — grouped attribution
    E_vanilla:  (window, features) — full per-feature attribution
    groups:     {name: [indices]} dict
    """
    if not groups:
        a = np.abs(E_adaptive).mean(axis=0)  # per-feature mean attribution
        b = np.abs(E_vanilla ).mean(axis=0)
        rho, _ = spearmanr(a, b)
        return float(rho)

    # Sum attribution within each group for both methods
    grp_adaptive, grp_vanilla = [], []
    for name, idxs in groups.items():
        grp_adaptive.append(np.abs(E_adaptive[:, idxs]).mean())
        grp_vanilla.append( np.abs(E_vanilla[:, idxs]).mean())

    if len(grp_adaptive) < 2:
        return float('nan')

    rho, _ = spearmanr(grp_adaptive, grp_vanilla)
    return float(rho)


print('✅ AOPC and group-rank Spearman ρ defined')

In [ ]:
# ── H3. Compute fidelity metrics for all N_EVAL steps ─────────────────────
print('Computing fidelity metrics (this takes a moment)...')

def evaluate_fidelity(stream, model_fn, E_vanilla_list, results_list, groups,
                       n_eval, label):
    aopc_vanilla   = []
    aopc_adaptive  = []
    rho_group      = []

    # Rebuild window for each step
    buf = deque(maxlen=WINDOW)

    for i in range(n_eval):
        x = stream[i]
        buf.append(x)
        W = np.array(buf)
        if len(W) < WINDOW:
            W = np.vstack([np.zeros((WINDOW-len(W), x.shape[0])), W])

        E_van = E_vanilla_list[i]
        E_adp = results_list[i].E

        # AOPC
        try:
            aopc_vanilla.append(aopc(model_fn, W, E_van, k_max=8))
            aopc_adaptive.append(aopc(model_fn, W, E_adp, k_max=8))
        except Exception:
            pass

        # Group-rank Spearman ρ
        rho = group_rank_spearman(E_adp, E_van, groups)
        if not np.isnan(rho):
            rho_group.append(rho)

    print(f'\n[{label}]')
    print(f'  AOPC Vanilla:          {np.mean(aopc_vanilla):.5f}  (higher = better faithfulness)')
    print(f'  AOPC Adaptive:         {np.mean(aopc_adaptive):.5f}')
    print(f'  AOPC ratio (adp/van):  {np.mean(aopc_adaptive)/max(np.mean(aopc_vanilla),1e-9):.3f}')
    print(f'  Group-rank Spearman ρ: {np.mean(rho_group):.3f}  (correct fidelity measure)')
    return aopc_vanilla, aopc_adaptive, rho_group


aopc_van_t, aopc_adp_t, rho_t = evaluate_fidelity(
    stream_temp, model_temp, E_van_temp, results_temp,
    temp_groups, N_EVAL, 'Temperature'
)
aopc_van_p, aopc_adp_p, rho_p = evaluate_fidelity(
    stream_power_norm, model_power, E_van_power, results_power,
    power_groups, N_EVAL, 'Power'
)

## I. Results Visualisation

In [ ]:
# ── I1. Master results figure ──────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('AdaptiveDeltaXAI Validation Results — DC Telemetry', fontsize=15, fontweight='bold')

# ── Row 0: Temperature explanations ───────────────────────────────────────
# Attribution heatmap — Vanilla
ax = fig.add_subplot(gs[0, 0])
E_stack_van_t = np.array([e.mean(axis=0) for e in E_van_temp])  # (N_EVAL, features)
im = ax.imshow(E_stack_van_t.T, aspect='auto', cmap='RdBu_r',
               vmax=np.abs(E_stack_van_t).max(), vmin=-np.abs(E_stack_van_t).max())
plt.colorbar(im, ax=ax, fraction=0.05)
ax.set_title('Temp — Vanilla SWING\n(attribution per feature over time)', fontsize=9)
ax.set_xlabel('Timestep'); ax.set_ylabel('Sensor')
ax.set_yticks(range(N_FEAT_TEMP)); ax.set_yticklabels(temp_cols, fontsize=7)

# Attribution heatmap — Adaptive
ax = fig.add_subplot(gs[0, 1])
E_stack_adp_t = np.array([r.E.mean(axis=0) for r in results_temp])
im = ax.imshow(E_stack_adp_t.T, aspect='auto', cmap='RdBu_r',
               vmax=np.abs(E_stack_adp_t).max(), vmin=-np.abs(E_stack_adp_t).max())
plt.colorbar(im, ax=ax, fraction=0.05)
# Mark recompute steps
for r in results_temp:
    if r.triggered:
        ax.axvline(r.t - 1, color='orange', lw=0.5, alpha=0.7)
ax.set_title('Temp — Adaptive\n(orange lines = recompute events)', fontsize=9)
ax.set_xlabel('Timestep'); ax.set_ylabel('Sensor')
ax.set_yticks(range(N_FEAT_TEMP)); ax.set_yticklabels(temp_cols, fontsize=7)

# Temperature: latency comparison
ax = fig.add_subplot(gs[0, 2])
ax.plot(lat_van_temp[:N_EVAL], color='#E05252', lw=0.8, label='Vanilla (every step)')
ax.plot([r.latency_ms for r in results_temp], color='#2E8B57', lw=0.8, label='Adaptive')
ax.set_title('Temperature — Latency per step (ms)', fontsize=9)
ax.set_xlabel('Timestep'); ax.set_ylabel('ms')
ax.legend(fontsize=8); ax.set_yscale('log')

# ── Row 1: Power explanations ─────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
E_stack_van_p = np.array([e.mean(axis=0) for e in E_van_power])
im = ax.imshow(E_stack_van_p.T, aspect='auto', cmap='RdBu_r',
               vmax=np.abs(E_stack_van_p).max(), vmin=-np.abs(E_stack_van_p).max())
plt.colorbar(im, ax=ax, fraction=0.05)
ax.set_title('Power — Vanilla SWING', fontsize=9)
ax.set_xlabel('Timestep'); ax.set_ylabel('Server')
ax.set_yticks(range(N_FEAT_POWER)); ax.set_yticklabels(power_cols, fontsize=7)

ax = fig.add_subplot(gs[1, 1])
E_stack_adp_p = np.array([r.E.mean(axis=0) for r in results_power])
im = ax.imshow(E_stack_adp_p.T, aspect='auto', cmap='RdBu_r',
               vmax=np.abs(E_stack_adp_p).max(), vmin=-np.abs(E_stack_adp_p).max())
plt.colorbar(im, ax=ax, fraction=0.05)
for r in results_power:
    if r.triggered:
        ax.axvline(r.t - 1, color='orange', lw=0.5, alpha=0.7)
ax.set_title('Power — Adaptive', fontsize=9)
ax.set_xlabel('Timestep'); ax.set_ylabel('Server')
ax.set_yticks(range(N_FEAT_POWER)); ax.set_yticklabels(power_cols, fontsize=7)

ax = fig.add_subplot(gs[1, 2])
ax.plot(lat_van_power[:N_EVAL], color='#E05252', lw=0.8, label='Vanilla')
ax.plot([r.latency_ms for r in results_power], color='#2E5EA5', lw=0.8, label='Adaptive')
ax.set_title('Power — Latency per step (ms)', fontsize=9)
ax.set_xlabel('Timestep'); ax.set_ylabel('ms')
ax.legend(fontsize=8); ax.set_yscale('log')

# ── Row 2: Fidelity metrics ────────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 0])
ax.plot(aopc_van_t, color='#E05252', lw=0.8, label='Vanilla')
ax.plot(aopc_adp_t, color='#2E8B57', lw=0.8, label='Adaptive')
ax.axhline(0, color='black', lw=0.5, ls='--')
ax.set_title('Temperature AOPC\n(higher = better faithfulness)', fontsize=9)
ax.set_xlabel('Timestep'); ax.legend(fontsize=8)

ax = fig.add_subplot(gs[2, 1])
ax.plot(aopc_van_p, color='#E05252', lw=0.8, label='Vanilla')
ax.plot(aopc_adp_p, color='#2E5EA5', lw=0.8, label='Adaptive')
ax.axhline(0, color='black', lw=0.5, ls='--')
ax.set_title('Power AOPC\n(higher = better faithfulness)', fontsize=9)
ax.set_xlabel('Timestep'); ax.legend(fontsize=8)

# Summary bar chart
ax = fig.add_subplot(gs[2, 2])
metrics = ['Recompute rate', 'Mean latency (ms)', 'AOPC', 'Group ρ']
van_t_vals = [100, np.mean(lat_van_temp[:N_EVAL]), np.mean(aopc_van_t), 1.0]
adp_t_vals = [sm_t['recompute_pct'], sm_t['lat_mean'], np.mean(aopc_adp_t), np.mean(rho_t) if rho_t else 0]
x = np.arange(len(metrics))
w = 0.35
ax.bar(x - w/2, van_t_vals, w, label='Vanilla', color='#E05252', alpha=0.8)
ax.bar(x + w/2, adp_t_vals, w, label='Adaptive', color='#2E8B57', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=7, rotation=15)
ax.set_title('Temperature — Summary comparison', fontsize=9)
ax.legend(fontsize=8)

plt.savefig('validation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Validation figure saved')

## J. Validation Checklist

In [ ]:
# ── J1. Automated validation checklist ────────────────────────────────────
print('=' * 65)
print('VALIDATION CHECKLIST — AdaptiveDeltaXAI vs VanillaDeltaXAI')
print('=' * 65)

checks = []

def check(label, cond, expected, actual_str):
    status = '✅ PASS' if cond else '❌ FAIL'
    print(f'  {status}  {label}')
    print(f'         Expected: {expected}  |  Got: {actual_str}')
    checks.append(cond)

# 1. DeltaSHAP repo cloned
check('DeltaSHAP repo cloned from AITRICS',
      os.path.isdir('DeltaSHAP'),
      'Directory exists',
      'exists' if os.path.isdir('DeltaSHAP') else 'missing')

# 2. DC datasets non-empty
check('DC Temperature dataset generated',
      len(stream_temp) > 100,
      '> 100 rows',
      f'{len(stream_temp)} rows')

check('DC Power dataset generated',
      len(stream_power_norm) > 100,
      '> 100 rows',
      f'{len(stream_power_norm)} rows')

# 3. Anomaly bursts present
anom_rate_t = df_temp['anomaly'].mean()
anom_rate_p = df_power['anomaly'].mean()
check('Temperature anomaly rate 5-40%',
      0.05 < anom_rate_t < 0.40,
      '5-40%',
      f'{anom_rate_t:.1%}')
check('Power anomaly rate 5-40%',
      0.05 < anom_rate_p < 0.40,
      '5-40%',
      f'{anom_rate_p:.1%}')

# 4. Vanilla baseline ran successfully
check('Vanilla SWING produced explanations (Temperature)',
      len(E_van_temp) == N_EVAL,
      f'{N_EVAL} explanations',
      f'{len(E_van_temp)}')
check('Vanilla SWING produced explanations (Power)',
      len(E_van_power) == N_EVAL,
      f'{N_EVAL} explanations',
      f'{len(E_van_power)}')

# 5. Adaptive recompute < vanilla (always 100%)
check('AdaptiveDeltaXAI recompute rate < 100% (Temperature)',
      sm_t['recompute_pct'] < 100,
      '< 100%',
      f"{sm_t['recompute_pct']:.1f}%")
check('AdaptiveDeltaXAI recompute rate < 100% (Power)',
      sm_p['recompute_pct'] < 100,
      '< 100%',
      f"{sm_p['recompute_pct']:.1f}%")

# 6. Adaptive latency faster than vanilla
van_lat_mean_t = np.mean(lat_van_temp[:N_EVAL])
check('Adaptive mean latency < Vanilla mean latency (Temperature)',
      sm_t['lat_mean'] < van_lat_mean_t,
      f'< {van_lat_mean_t:.2f}ms',
      f"{sm_t['lat_mean']:.2f}ms")
van_lat_mean_p = np.mean(lat_van_power[:N_EVAL])
check('Adaptive mean latency < Vanilla mean latency (Power)',
      sm_p['lat_mean'] < van_lat_mean_p,
      f'< {van_lat_mean_p:.2f}ms',
      f"{sm_p['lat_mean']:.2f}ms")

# 7. AOPC finite
check('AOPC scores computed (Temperature)',
      len(aopc_van_t) > 0 and not np.isnan(np.mean(aopc_van_t)),
      'Non-NaN values',
      f'mean={np.mean(aopc_van_t):.5f}')
check('AOPC scores computed (Power)',
      len(aopc_van_p) > 0 and not np.isnan(np.mean(aopc_van_p)),
      'Non-NaN values',
      f'mean={np.mean(aopc_van_p):.5f}')

# 8. Group-rank Spearman ρ valid
rho_t_mean = np.mean(rho_t) if rho_t else float('nan')
rho_p_mean = np.mean(rho_p) if rho_p else float('nan')
check('Group-rank Spearman ρ computed (Temperature)',
      len(rho_t) > 0,
      '> 0 valid ρ values',
      f'{len(rho_t)} values, mean={rho_t_mean:.3f}')
check('Group-rank Spearman ρ computed (Power)',
      len(rho_p) > 0,
      '> 0 valid ρ values',
      f'{len(rho_p)} values, mean={rho_p_mean:.3f}')

print('\n' + '=' * 65)
n_pass = sum(checks)
n_total = len(checks)
print(f'RESULT: {n_pass}/{n_total} checks passed')
print('=' * 65)

if n_pass == n_total:
    print('\n✅ ALL CHECKS PASSED — Validation complete')
else:
    print(f'\n⚠️  {n_total - n_pass} check(s) failed — review above')

In [ ]:
# ── J2. Final summary table ────────────────────────────────────────────────
print('\n' + '='*75)
print('MASTER RESULTS TABLE')
print('='*75)
print(f'{"Metric":<40} {"Temp Vanilla":>12} {"Temp Adaptive":>13} {"Power Vanilla":>13} {"Power Adaptive":>14}')
print('-'*92)

rows = [
    ('Recompute rate (%)',        100.0,           sm_t['recompute_pct'],   100.0,           sm_p['recompute_pct']),
    ('Mean latency (ms/step)',    van_lat_mean_t,  sm_t['lat_mean'],        van_lat_mean_p,  sm_p['lat_mean']),
    ('P95 latency (ms)',          np.percentile(lat_van_temp[:N_EVAL],95), sm_t['lat_p95'], np.percentile(lat_van_power[:N_EVAL],95), sm_p['lat_p95']),
    ('P99 latency (ms)',          np.percentile(lat_van_temp[:N_EVAL],99), sm_t['lat_p99'], np.percentile(lat_van_power[:N_EVAL],99), sm_p['lat_p99']),
    ('Mean cache age (steps)',    0.0,             sm_t['cache_age_mean'],  0.0,             sm_p['cache_age_mean']),
    ('Max cache age (steps)',     1,               sm_t['cache_age_max'],   1,               sm_p['cache_age_max']),
    ('AOPC (mean)',               np.mean(aopc_van_t), np.mean(aopc_adp_t), np.mean(aopc_van_p), np.mean(aopc_adp_p)),
    ('Group-rank Spearman ρ',    1.0,             rho_t_mean,              1.0,             rho_p_mean),
    ('Speedup vs vanilla',        1.0,             van_lat_mean_t/max(sm_t['lat_mean'],1e-6), 1.0, van_lat_mean_p/max(sm_p['lat_mean'],1e-6)),
]

for label, tv, ta, pv, pa in rows:
    def fmt(v):
        try: return f'{v:.2f}'
        except: return str(v)
    print(f'  {label:<38} {fmt(tv):>12} {fmt(ta):>13} {fmt(pv):>13} {fmt(pa):>14}')

print('='*92)
print('\n† Speedup = vanilla_latency / adaptive_latency (higher = faster adaptive)')
print('† Group-rank Spearman ρ is the correct fidelity metric for grouped attribution')
print('  (replaces raw Pearson ρ used in our Month 1-2 doc — that was incorrect)')

## Summary & What to Do Next

### Repository reality check
| Item | Finding |
|------|--------|
| `AITRICS/Delta-XAI` | ❌ Does not exist (paper under review) |
| `AITRICS/DeltaSHAP` | ✅ Cloned & audited above — workshop predecessor |
| Anonymous review code | 🔒 Read-only, linked from OpenReview |
| Our AdaptiveDeltaXAI | ✅ Faithful reconstruction from paper methods |

### Dataset choices
| Dataset | Status | Notes |
|---------|--------|-------|
| P19 PhysioNet | ✅ Public (figshare link) | 34 sensors, sepsis labels |
| MIMIC-III | 🔒 Credentialed | Used by DeltaSHAP, requires PhysioNet signup |
| Synthetic DC Temp | ✅ Generated here | Thermal physics model |
| Synthetic DC Power | ✅ Generated here | PDU coupling, periodic workload |

### Key metric corrections
- ❌ Raw Pearson ρ (Month 1-2 doc) — wrong for grouped vs full-feature attribution
- ✅ **Group-rank Spearman ρ** — correct fidelity when using grouped attribution
- ✅ **AOPC** — the paper's own faithfulness metric, now implemented here

### Next steps
1. Get MIMIC-III access via PhysioNet and run DeltaSHAP natively
2. Increase `N_STEPS=20` for publication-quality IG integration
3. Train the GRU anomaly model (currently random weights) on real DC data
4. Add ShaTS graph-based grouping (rack adjacency matrix)
5. Run CausalRCA post-processing on E_t outputs